In [1]:
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup
import re

In [2]:
df = pd.read_excel("../data/myfair_participation_message_1982.xlsx").dropna(how='all')
df.head(3)

,id,message,process_state,sender_email,message_type,participation_num
0,43031.0,"<div class=""css-0""><br data-mce-bogus=""1""></di...",부스 예약 접수 완료,customer@customer.com,ADMIN,1982
1,43034.0,"<p>(고객사 담당자명 직함)님, 안녕하세요</p><p>마이페어 (마이페어 담당자명...",참가 품목 안내,myfair@myfair.co,ADMIN,1982
2,43435.0,"<p>안녕하세요, 마이페어입니다.</p><p><br data-mce-bogus=""1...",부스 포함사항 안내,myfair@myfair.co,ADMIN,1982


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171 entries, 0 to 170
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 165 non-null    float64
 1   message            165 non-null    object 
 2   process_state      81 non-null     object 
 3   sender_email       163 non-null    object 
 4   message_type       165 non-null    object 
 5   participation_num  171 non-null    int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 8.1+ KB


# 영문 번역 라이브러리: Googletrans

In [4]:
from googletrans import Translator
translator = Translator()
result = translator.translate('안녕하세요', src='ko', dest='en')
print(result.text)  # "Hello"

hello


In [5]:
result = translator.translate('안녕하세요 저는 홍길동입니다. 만나서 반갑습니다.', src='ko', dest='en')
print(result.text)  # "Hello"

Hello, I am Hong Gil -dong.nice to meet you


# 전처리

In [6]:
def preprocess_html_message(html_text):
    if pd.isna(html_text):
        return ""  # 또는 NaN 유지하려면: return np.nan

    # BeautifulSoup으로 HTML 파싱
    soup = BeautifulSoup(html_text, "html.parser")

    # <br> 태그는 줄바꿈으로 대체
    for br in soup.find_all("br"):
        br.replace_with("\n")

    # 텍스트만 추출
    raw_text = soup.get_text(separator=" ", strip=True)

    # 공백 정리
    raw_text = re.sub(r'\s+', ' ', raw_text)

    return raw_text

In [7]:
df['message_clean'] = df['message'].apply(preprocess_html_message)

# 결과 출력
print(df[['message', 'message_clean']].head())

                                             message  \
0  <div class="css-0"><br data-mce-bogus="1"></di...   
1  <p>(고객사 담당자명 직함)님, 안녕하세요</p><p>마이페어 (마이페어 담당자명...   
2  <p>안녕하세요, 마이페어입니다.</p><p><br data-mce-bogus="1...   
3  <p>담당자님 안녕하세요 보내주신 내용 확인했습니다<br>부재중이셔서 채팅드립니다<...   
4  <p>네 담당자님 확인했습니다!&nbsp;혹시 부스 자리 선택 또는 2면 오픈 부스...   

                                       message_clean  
0  🎉 부스예약이 접수되었습니다. 마이페어에서 참가신청을 시작해 주셔서 감사합니다. 서...  
1  (고객사 담당자명 직함)님, 안녕하세요 마이페어 (마이페어 담당자명)입니다. 비품이...  
2  안녕하세요, 마이페어입니다. 답변 기다려 주셔서 감사합니다. 부스패키지는 아래와 같...  
3  담당자님 안녕하세요 보내주신 내용 확인했습니다 부재중이셔서 채팅드립니다 부스 안에 ...  
4  네 담당자님 확인했습니다! 혹시 부스 자리 선택 또는 2면 오픈 부스 선택 가능한가...  


In [8]:
df['message_clean'][150]

'안녕하세요, (고객사 담당자명 직함)님. 마이페어 (마이페어 담당자명 호칭)입니다. 네! 지금 바로 하실 수 있게 도와드렸습니다! 해당 절차 이후, 별도로 해주실 건 없으시며 정산에 필요한 요청 드렸던 서류 전달 주시면 됩니다. 이후로는 저희가 정산기관에 모든 서류들 모아서 제출하고 정산 받겠습니다. (약 1~2개월 소요) 감사합니다.'

# message 컬럼 번역

In [9]:
from googletrans import Translator

# Translator 객체 생성
translator = Translator()

# message 컬럼에 None 값이 섞여 있다면 미리 제거
df['message'] = df['message'].fillna('')

# 결측치 제외하고 번역
def translate_text(text):
    try:
        return translator.translate(text, src='ko', dest='en').text
    except Exception as e:
        print(f"Translation failed: {e}")
        return text  # 번역 실패 시 원문 반환

# message 컬럼 영어로 번역한 새 컬럼 추가
df['message_en'] = df['message_clean'].apply(lambda x: translate_text(x) if pd.notnull(x) else x)

Translation failed: the JSON object must be str, bytes or bytearray, not NoneType
Translation failed: the JSON object must be str, bytes or bytearray, not NoneType
Translation failed: the JSON object must be str, bytes or bytearray, not NoneType
Translation failed: the JSON object must be str, bytes or bytearray, not NoneType
Translation failed: the JSON object must be str, bytes or bytearray, not NoneType
Translation failed: the JSON object must be str, bytes or bytearray, not NoneType


In [10]:
# 결과 출력
print(df[['message_clean', 'message_en']].head())

                                       message_clean  \
0  🎉 부스예약이 접수되었습니다. 마이페어에서 참가신청을 시작해 주셔서 감사합니다. 서...   
1  (고객사 담당자명 직함)님, 안녕하세요 마이페어 (마이페어 담당자명)입니다. 비품이...   
2  안녕하세요, 마이페어입니다. 답변 기다려 주셔서 감사합니다. 부스패키지는 아래와 같...   
3  담당자님 안녕하세요 보내주신 내용 확인했습니다 부재중이셔서 채팅드립니다 부스 안에 ...   
4  네 담당자님 확인했습니다! 혹시 부스 자리 선택 또는 2면 오픈 부스 선택 가능한가...   

                                          message_en  
0  🎉 booth booking has been received.Thank you fo...  
1  (Customer personnel), Hello, this is My Fair (...  
2  Hello, this is My Fair.Thank you for waiting f...  
3         Hello, I checked the contents you sent me.  
4  You have checked your person in charge!Is it p...  


In [11]:
df['message_en'][150]

'Hello, (Title of the customer).My Fair is the name of My Fair.yesI helped you to do it right now!After this procedure, you have nothing to do with you and deliver the documents you needed for settlement.Since then, we will collect and submit all the documents to the settlement agency.Thank you.'

In [12]:
df

,id,message,process_state,sender_email,message_type,participation_num,message_clean,message_en
0,43031.0,"<div class=""css-0""><br data-mce-bogus=""1""></di...",부스 예약 접수 완료,customer@customer.com,ADMIN,1982,🎉 부스예약이 접수되었습니다. 마이페어에서 참가신청을 시작해 주셔서 감사합니다. 서...,🎉 booth booking has been received.Thank you fo...
1,43034.0,"<p>(고객사 담당자명 직함)님, 안녕하세요</p><p>마이페어 (마이페어 담당자명...",참가 품목 안내,myfair@myfair.co,ADMIN,1982,"(고객사 담당자명 직함)님, 안녕하세요 마이페어 (마이페어 담당자명)입니다. 비품이...","(Customer personnel), Hello, this is My Fair (..."
2,43435.0,"<p>안녕하세요, 마이페어입니다.</p><p><br data-mce-bogus=""1...",부스 포함사항 안내,myfair@myfair.co,ADMIN,1982,"안녕하세요, 마이페어입니다. 답변 기다려 주셔서 감사합니다. 부스패키지는 아래와 같...","Hello, this is My Fair.Thank you for waiting f..."
3,43516.0,<p>담당자님 안녕하세요 보내주신 내용 확인했습니다<br>부재중이셔서 채팅드립니다<...,NaN,customer@customer.com,CUSTOMER,1982,담당자님 안녕하세요 보내주신 내용 확인했습니다 부재중이셔서 채팅드립니다 부스 안에 ...,"Hello, I checked the contents you sent me."
4,43532.0,<p>네 담당자님 확인했습니다!&nbsp;혹시 부스 자리 선택 또는 2면 오픈 부스...,NaN,customer@customer.com,CUSTOMER,1982,네 담당자님 확인했습니다! 혹시 부스 자리 선택 또는 2면 오픈 부스 선택 가능한가...,You have checked your person in charge!Is it p...
...,...,...,...,...,...,...,...,...
166,NaN,,NaN,NaN,NaN,1982,,
167,NaN,,NaN,NaN,NaN,1982,,
168,NaN,,NaN,NaN,NaN,1982,,
169,NaN,,NaN,NaN,NaN,1982,,


In [13]:
missing_id_rows = df[df['id'].isna()]
print(missing_id_rows)

     id message process_state sender_email message_type  participation_num  \
165 NaN                   NaN          NaN          NaN               1982   
166 NaN                   NaN          NaN          NaN               1982   
167 NaN                   NaN          NaN          NaN               1982   
168 NaN                   NaN          NaN          NaN               1982   
169 NaN                   NaN          NaN          NaN               1982   
170 NaN                   NaN          NaN          NaN               1982   

    message_clean message_en  
165                           
166                           
167                           
168                           
169                           
170                           


In [14]:
df = df.drop(index=range(165, 171))  # 171은 포함되지 않음

In [15]:
df.isna().sum()

id                    0
message               0
process_state        84
sender_email          2
message_type          0
participation_num     0
message_clean         0
message_en            0
dtype: int64

In [16]:
df['message_en']

0      🎉 booth booking has been received.Thank you fo...
1      (Customer personnel), Hello, this is My Fair (...
2      Hello, this is My Fair.Thank you for waiting f...
3             Hello, I checked the contents you sent me.
4      You have checked your person in charge!Is it p...
                             ...                        
160                  Thank you, please contact me again!
161    Hello, (Title of the customer).My Fair is the ...
162    Hello, (Title of the customer).My Fair is the ...
163    Thank you for your late time!Thank you so kind...
164    Hello, (Title of the customer).My Fair is the ...
Name: message_en, Length: 165, dtype: object

***

# 비식별화

1. 마이페어 상담원 비식별화
- 현재 조건: "마이페어 홍길동입니다" → "마이페어 (마이페어 담당자명)입니다"

> 마이페어 담당자들은 대부분 "마이페어 + 이름 + 입니다" 패턴을 사용함

- 추가로 고려하면 좋은 점: "마이페어 홍길동 드림", "마이페어 김지훈 올림" 같은 변형 및 +"마이페어 홍길동입니다~" 처럼 붙어있는 형태

2. 고객사 담당자 비식별화
- 현재 조건: "홍길동 님"이나 "김부장님"을 → (고객사 담당자명 직함)님

> 문제점: 고객도 마이페어 직원에게 “홍길동 님”처럼 부를 수 있어 → 양방향 모두에 “님” 사용, 따라서 “님”만 보고 누가 누구인지 구분이 불가능

### 해결 전략
발화자의  sender_email 정보를 고려하자:
- 마이페어 이메일 도메인으로부터 발신자 역할(sender role) 추정 가능
- 예: sender_email이 @myfair.co.kr → 마이페어 직원
- 이 정보를 바탕으로 누가 말했는지 구분

In [29]:
def anonymize_message_v2(text, sender_email):
    if pd.isna(text):
        return text

    # 마이페어 상담원이 보낸 경우
    if "@myfair.co.kr" in sender_email:
        # 마이페어 + 이름 → 마이페어 (마이페어 담당자명)
        text = re.sub(r"마이페어\s[가-힣]{2,}(?=입니다|입니다\.)", "마이페어 (마이페어 담당자명)", text)

        # 고객사 담당자명 추정되는 "~~님"만 치환 (첫 번째 1회만)
        text = re.sub(r"[가-힣]{2,}(?:\s?[가-힣]{2,})?님", "(고객사 담당자명 직함)님", text, count=1)

    else:  # 고객사 담당자가 보낸 경우
        # 마이페어 상담원 이름이 있는 경우 → (마이페어 담당자명)으로 대체
        text = re.sub(r"마이페어\s[가-힣]{2,}님", "마이페어 (마이페어 담당자명)님", text)

    return text

In [31]:
ex = pd.DataFrame({
    "sender_email": [
        "agent01@myfair.co.kr",  # 마이페어 상담원
        "client@company.com",    # 고객사 담당자
        "agent02@myfair.co.kr",  # 마이페어 상담원
        "client@company.com",    # 고객사 담당자
    ],
    "message": [
        "김부장님, 안녕하세요. 마이페어 김지훈입니다.",
        "마이페어 김지훈님, 안녕하세요. 궁금한 게 있습니다.",
        "박과장님, 일정 확인 부탁드립니다. 마이페어 이민호입니다.",
        "마이페어 이민호님, 감사합니다!"
    ]
})

# 적용
ex['message_anonymized'] = ex.apply(
    lambda row: anonymize_message_v2(row['message'], row['sender_email']),
    axis=1
)

In [32]:
# 출력 확인
print(ex[['message', 'message_anonymized']])

                            message  \
0         김부장님, 안녕하세요. 마이페어 김지훈입니다.   
1     마이페어 김지훈님, 안녕하세요. 궁금한 게 있습니다.   
2  박과장님, 일정 확인 부탁드립니다. 마이페어 이민호입니다.   
3                 마이페어 이민호님, 감사합니다!   

                                  message_anonymized  
0        (고객사 담당자명 직함)님, 안녕하세요. 마이페어 (마이페어 담당자명)입니다.  
1              마이페어 (마이페어 담당자명)님, 안녕하세요. 궁금한 게 있습니다.  
2  (고객사 담당자명 직함)님, 일정 확인 부탁드립니다. 마이페어 (마이페어 담당자명)...  
3                          마이페어 (마이페어 담당자명)님, 감사합니다!  


***

# 다른 데이터에 대해 적용

In [26]:
df2 = pd.read_excel("../data/myfair_participation_message_2268.xlsx").dropna(how='all')
df2.head(3)

,id,message,process_state,sender_email,message_type,participation_num
0,84746.0,"<p style="""">안녕하세요, (고객 성함 직함)님.</p><p style=""""...",독립부스 시공 서비스 안내,myfair01@myfair.co,ADMIN,2268
1,84760.0,"<p style="""">프로님 안녕하세요. (고객사명) (고객 성함)입니다.</p><...",NaN,customer@customer.com,CUSTOMER,2268
2,84799.0,"<p style="""">안녕하세요, (고객 성함 직함)님.</p><p style=""""...",문의 답변,myfair01@myfair.co,ADMIN,2268
